# Aggregation Strategy 1: Mean Pool

The zero-parameter baseline. Every camera view contributes equally to the final feature vector.

## Formulation

Given per-view features $\{v_1, v_2, \dots, v_V\}$ with $v_i \in \mathbb{R}^d$ extracted by the shared backbone, mean pool computes:

$$z = \frac{1}{V} \sum_{i=1}^{V} v_i$$

The classification heads then consume the single pooled vector $z \in \mathbb{R}^d$.



## Code

The live implementation lives in [`VARS model/mvaggregate.py`](../VARS%20model/mvaggregate.py) as `ViewAvgAggregate`. The block below imports that class so this notebook stays in sync with the code that actually trains.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../VARS model'))

import torch
from torch import nn
from mvaggregate import ViewAvgAggregate

print('ViewAvgAggregate imported from:', ViewAvgAggregate.__module__)

/venv/vars/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ViewAvgAggregate imported from: mvaggregate


## Forward-pass demo on dummy features

We bypass the backbone and demonstrate the aggregation math directly on pre-extracted view features of shape `(B, V, feat_dim)`. This is the exact operation `ViewAvgAggregate` performs in its last step.

In [2]:
torch.manual_seed(0)
B, V, feat_dim = 2, 3, 512
view_features = torch.randn(B, V, feat_dim)
print('Per-view features:', view_features.shape)

# The core aggregation step — single line.
pooled = view_features.mean(dim=1)
print('Pooled feature:   ', pooled.shape)

Per-view features: torch.Size([2, 3, 512])
Pooled feature:    torch.Size([2, 512])


In [3]:
# Sanity check: pooled is the arithmetic mean of the views.
manual = (view_features[:, 0] + view_features[:, 1] + view_features[:, 2]) / 3
assert torch.allclose(pooled, manual)
print('Mean pool matches manual average ✓')

Mean pool matches manual average ✓


## Parameter count

Mean pool has no parameters of its own — but `ViewAvgAggregate` does hold a reference to the shared backbone. Parameter count for the *aggregation block alone* (excluding the backbone) is the fair comparison metric.

In [4]:
class IdentityBackbone(nn.Module):
    def forward(self, x):
        return x

agg = ViewAvgAggregate(model=IdentityBackbone())
own_params = sum(p.numel() for name, p in agg.named_parameters() if not name.startswith('model.'))
print(f'Mean pool aggregation block parameters: {own_params}')

Mean pool aggregation block parameters: 0
